In [ ]:
import os
import numpy as np
import pandas as pd
import textwrap
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
def load_sensor_and_curve_data(sensor_data_path, pump_curve_path,
                                skiprows=3, slice_start=40000, slice_end=41000):
    """Load raw Excel sensor data + pump curve Excel.

    Returns
    -------
    Time : np.ndarray[datetime64]
    Elapsed_time : np.ndarray[float]   (seconds from first sample)
    PIP_signal : np.ndarray[float]     (atm)
    PDP_signal : np.ndarray[float]     (atm)
    pump_Q : np.ndarray[float]
    pump_Head : np.ndarray[float]
    """
    sensor_df = pd.read_excel(sensor_data_path, skiprows=skiprows)
    sensor_df = sensor_df.iloc[slice_start:slice_end].copy()
    curve_df = pd.read_excel(pump_curve_path)

    sensor_df.columns = sensor_df.columns.astype(str).str.strip()
    curve_df.columns = curve_df.columns.astype(str).str.strip()

    sensor_df["DateTime"] = pd.to_datetime(
        sensor_df["Date"].astype(str) + " " + sensor_df["Time"].astype(str),
        errors="coerce",
    )
    sensor_df["Elapsed_time_sec"] = (
        sensor_df["DateTime"] - sensor_df["DateTime"].iloc[0]
    ).dt.total_seconds()

    sensor_df = sensor_df.dropna(
        subset=["DateTime", "Elapsed_time_sec", "Pamb, atm", "Pump dischar.pres, atm"]
    ).reset_index(drop=True)

    Time = sensor_df["DateTime"].to_numpy()
    Elapsed_time = sensor_df["Elapsed_time_sec"].to_numpy()
    PIP_signal = sensor_df["Pamb, atm"].to_numpy()
    PDP_signal = sensor_df["Pump dischar.pres, atm"].to_numpy()

    Q = curve_df["Rate"].to_numpy()
    Head = curve_df["Head"].to_numpy()
    Power = curve_df["Power"].to_numpy()
    Efficiency =  curve_df["Efficiency"].to_numpy()
    return Time, Elapsed_time, PIP_signal, PDP_signal, Q, Head, Power, Efficiency
def atm_to_meters(pressure_atm, SG):
    """atm -> meters of fluid column with specific gravity SG."""
    return pressure_atm * 10.33 / SG

def atm_to_psi(pressure_atm):
    """atm -> psi. 1 atm = 14.6959 psi."""
    return pressure_atm * 14.6959

def head_meters_to_psi(head_m, SG):
    """Head in meters of fluid column -> pressure in psi."""
    return (head_m * SG) / 0.70307

def psi_to_head_meters(psi, SG):
    """Pressure in psi -> head in meters of fluid column."""
    return (psi * 0.70307) / SG

def m3day_to_bblday(q_m3day):
    """Flow rate: m3/day -> bbl/day. 1 m3 = 6.28981 bbl."""
    return q_m3day * 6.28981

def bblday_to_m3day(q_bblday):
    """Flow rate: bbl/day -> m3/day."""
    return q_bblday / 6.28981
def apply_affinity_scaling(Q, Head, Power, f_ratio):
    """Centrifugal pump affinity laws: Head ~ f^2, Q ~ f, Power ~ f^3.
    Efficiency unchanged at corresponding points.
    """
    Q_af = Q * f_ratio
    Head_af = Head * (f_ratio ** 2)
    Power_af = Power * (f_ratio ** 3)
    return Q_af, Head_af, Power_af
def apply_SG_correction_to_power(Power, SG, SG_reference=0.9):
    """Correct the pump curve Power for a different fluid specific gravity.

    The digitized pump curve is assumed to have been measured with a
    reference fluid of specific gravity `SG_reference` (default 0.9).
    If the actual fluid has a different SG, the hydraulic power scales
    linearly with SG (because hydraulic power = ρ·g·Q·H, and ρ ∝ SG).

    Parameters
    ----------
    Power : np.ndarray or float
        Original pump curve Power values (at SG_reference).
    SG : float
        Actual fluid specific gravity.
    SG_reference : float, optional
        Reference SG the curve was digitized at. Default 0.9.

    Returns
    -------
    np.ndarray or float
        Corrected Power values. If SG == SG_reference, returns Power unchanged.
    """
    if SG == SG_reference:
        return Power
    return Power * (SG / SG_reference)
def build_interp_arrays(Q, Head, Power, Efficiency):
    """Sort curve arrays for both interpolation directions.

    Returns
    -------
    head_arr_for_q, q_arr_for_q : sorted by Head ascending  (Head -> Q)
    q_arr, eff_arr, power_arr   : sorted by Q ascending     (Q -> Eff, Power)
    """
    # Filter out non-finite
    valid = np.isfinite(Q) & np.isfinite(Head) & np.isfinite(Power) & np.isfinite(Efficiency)
    Q, Head, Power, Efficiency = Q[valid], Head[valid], Power[valid], Efficiency[valid]

    # Head -> Q direction
    head_sort_idx = np.argsort(Head)
    head_sorted = Head[head_sort_idx]
    q_for_head = Q[head_sort_idx]
    # remove duplicate head values
    head_unique, unique_idx = np.unique(head_sorted, return_index=True)
    q_for_head = q_for_head[unique_idx]
    head_arr_for_q = head_unique
    q_arr_for_q = q_for_head

    # Q -> Eff/Power direction
    q_sort_idx = np.argsort(Q)
    q_arr = Q[q_sort_idx]
    eff_arr = Efficiency[q_sort_idx]
    power_arr = Power[q_sort_idx]

    return head_arr_for_q, q_arr_for_q, q_arr, eff_arr, power_arr
def compute_operating_points(Elapsed_time, PIP_signal, PDP_signal,
                              head_arr_for_q, q_arr_for_q,
                              q_arr, eff_arr, power_arr,
                              observe_window=1.0, SG=1.0,
                              head_match_tol=1e-6,
                              pressure_already_in_meters=False):
    """Compute Head/Q/Efficiency/Power for every non-overlapping time window.

    Each sensor sample is assigned to exactly one observation window using
    the actual elapsed-time array. The window boundaries are deterministic
    and are not affected by floating-point accumulation in np.arange.

    Parameters
    ----------
    Elapsed_time : np.ndarray
        Elapsed seconds from the first sample.
    PIP_signal, PDP_signal : np.ndarray
        Raw pressure signals. If `pressure_already_in_meters=False` (default),
        these are in atm and will be converted using SG. If True, they are
        already in meters of fluid column.
    head_arr_for_q, q_arr_for_q : np.ndarray
        Curve arrays sorted by Head ascending.
    q_arr, eff_arr, power_arr : np.ndarray
        Curve arrays sorted by Q ascending.
    observe_window : float
        Window length in seconds.
    SG : float
        Specific gravity of the fluid (used only if pressure needs conversion).
    head_match_tol : float
        Absolute tolerance for "exact" Head match.
    pressure_already_in_meters : bool
        If True, skip atm->m conversion.

    Returns
    -------
    pandas.DataFrame with columns:
        Window_Start_s, Window_End_s, PIP, PDP, Head,
        Q, Q_match, Efficiency, Power, Status
    """
    if observe_window <= 0:
        raise ValueError("observe_window must be greater than zero.")

    # Convert to arrays and keep all three signals synchronized.
    Elapsed_time = np.asarray(Elapsed_time, dtype=float)
    PIP_signal = np.asarray(PIP_signal, dtype=float)
    PDP_signal = np.asarray(PDP_signal, dtype=float)

    valid_time = np.isfinite(Elapsed_time)
    Elapsed_time = Elapsed_time[valid_time]
    PIP_signal = PIP_signal[valid_time]
    PDP_signal = PDP_signal[valid_time]

    if Elapsed_time.size == 0:
        return pd.DataFrame(columns=[
            "Window_Start_s", "Window_End_s", "PIP", "PDP", "Head",
            "Head_psi", "Q", "Q_match", "Efficiency", "Power", "Status"
        ])

    # Make sure time is monotonically increasing before using searchsorted.
    sort_idx = np.argsort(Elapsed_time, kind="mergesort")
    Elapsed_time = Elapsed_time[sort_idx]
    PIP_signal = PIP_signal[sort_idx]
    PDP_signal = PDP_signal[sort_idx]

    Head_min, Head_max = head_arr_for_q[0], head_arr_for_q[-1]
    t_min = Elapsed_time[0]
    t_max = Elapsed_time[-1]

    results = []

    # Build window boundaries once. Each interval is [start, end), so a
    # sample exactly on a boundary belongs to the next window only.
    window_starts = np.arange(t_min, t_max + observe_window, observe_window)

    for start_time in window_starts:
        end_time = start_time + observe_window

        left = np.searchsorted(Elapsed_time, start_time, side="left")
        right = np.searchsorted(Elapsed_time, end_time, side="left")

        if right <= left:
            continue

        PIP_raw = np.nanmean(PIP_signal[left:right])
        PDP_raw = np.nanmean(PDP_signal[left:right])

        if not (np.isfinite(PIP_raw) and np.isfinite(PDP_raw)):
            continue

        if pressure_already_in_meters:
            PIP_m = PIP_raw
            PDP_m = PDP_raw
        else:
            PIP_m = atm_to_meters(PIP_raw, SG)
            PDP_m = atm_to_meters(PDP_raw, SG)

        Head = PDP_m - PIP_m

        # ---- Out-of-range check ----
        if not (Head_min <= Head <= Head_max):
            Q = np.nan
            match_type = "out_of_range"
            Efficiency = np.nan
            Power = np.nan
            status = "out_of_range"
        else:
            # ---- Head -> Q ----
            close_idx = np.where(
                np.isclose(head_arr_for_q, Head, atol=head_match_tol)
            )[0]
            if close_idx.size > 0:
                Q = q_arr_for_q[close_idx[0]]
                match_type = "exact"
            else:
                Q = np.interp(Head, head_arr_for_q, q_arr_for_q)
                match_type = "interpolated"

            # ---- Q -> Efficiency, Power ----
            Efficiency = np.interp(Q, q_arr, eff_arr)
            Power = np.interp(Q, q_arr, power_arr)
            status = "valid"

        results.append({
            "Window_Start_s": start_time,
            "Window_End_s": end_time,
            "PIP": PIP_m,
            "PDP": PDP_m,
            "Head": Head,
            "Head_psi": head_meters_to_psi(Head, SG),
            "Q": Q,
            "Q_match": match_type,
            "Efficiency": Efficiency,
            "Power": Power,
            "Status": status,
        })

    return pd.DataFrame(results)

def find_bep_and_safe_envelope(Q, Efficiency, envelope_frac=0.1):
    """Find BEP and the Recommended Operating Range (ROR)."""
    valid = np.isfinite(Q) & np.isfinite(Efficiency)
    Q, Efficiency = Q[valid], Efficiency[valid]

    # ---- Q_BEP: دستی روی 1300 ----
    Q_BEP = 1300.0

    # Efficiency at Q_BEP از منحنی درون‌یابی می‌شود
    q_sort_idx = np.argsort(Q)
    q_sorted = Q[q_sort_idx]
    eff_sorted = Efficiency[q_sort_idx]
    Efficiency_BEP = float(np.interp(Q_BEP, q_sorted, eff_sorted))

    # ROR: ثابت (650, 1900)
    Q_low = 650.0
    Q_high = 1900.0

    Efficiency_low_env = float(np.interp(Q_low, q_sorted, eff_sorted))
    Efficiency_high_env = float(np.interp(Q_high, q_sorted, eff_sorted))

    return {
        "Q_BEP": Q_BEP,
        "Efficiency_BEP": Efficiency_BEP,
        "Q_low": Q_low,
        "Q_high": Q_high,
        "Efficiency_low_env": Efficiency_low_env,
        "Efficiency_high_env": Efficiency_high_env,
    }
def check_operating_status(Q_point, Efficiency_point, bep_info):
    """Check a single operating point against ROR and BEP."""
    if not np.isfinite(Q_point) or not np.isfinite(Efficiency_point):
        return {
            "in_envelope": False,
            "side": None,
            "distance_Q": np.nan,
            "distance_pct": np.nan,
            "distance_from_BEP_Q": np.nan,
            "distance_from_BEP_pct": np.nan,
            "BEP_side": "Unknown",
            "efficiency_drop_from_BEP": np.nan,
            "efficiency_drop_pct": np.nan,
            "message": "WARNING: No valid operating point (Head outside pump curve range).",
        }

    Q_low = bep_info["Q_low"]
    Q_high = bep_info["Q_high"]
    Q_BEP = bep_info["Q_BEP"]
    Efficiency_BEP = bep_info["Efficiency_BEP"]

    distance_from_BEP_Q = abs(Q_point - Q_BEP)
    distance_from_BEP_pct = distance_from_BEP_Q / Q_BEP * 100

    if Q_point < Q_BEP:
        BEP_side = "Operating Below the BEP"
    elif Q_point > Q_BEP:
        BEP_side = "Operating Above the BEP"
    else:
        BEP_side = "Operating at the BEP"

    efficiency_drop_from_BEP = Efficiency_BEP - Efficiency_point
    efficiency_drop_pct = efficiency_drop_from_BEP / Efficiency_BEP * 100

    if Q_low <= Q_point <= Q_high:
        return {
            "in_envelope": True,
            "side": None,
            "distance_Q": 0.0,
            "distance_pct": 0.0,
            "distance_from_BEP_Q": distance_from_BEP_Q,
            "distance_from_BEP_pct": distance_from_BEP_pct,
            "BEP_side": BEP_side,
            "efficiency_drop_from_BEP": efficiency_drop_from_BEP,
            "efficiency_drop_pct": efficiency_drop_pct,
            "message": f"OK: Q={Q_point:.1f} is in the Recommended Operating Range (ROR).",
        }

    if Q_point < Q_low:
        distance_Q = Q_low - Q_point
        side = "low"
    else:
        distance_Q = Q_point - Q_high
        side = "high"

    distance_pct = distance_Q / Q_BEP * 100
    bound = Q_low if side == "low" else Q_high
    message = (
        f"WARNING: OUTSIDE the Recommended Operating Range (ROR), on the {side} side. "
        f"Q={Q_point:.1f} is {distance_Q:.1f} m3/day ({distance_pct:.1f}% of Q_BEP) "
        f"past the {'lower' if side == 'low' else 'upper'} bound ({bound:.1f})."
    )

    return {
        "in_envelope": False,
        "side": side,
        "distance_Q": distance_Q,
        "distance_pct": distance_pct,
        "distance_from_BEP_Q": distance_from_BEP_Q,
        "distance_from_BEP_pct": distance_from_BEP_pct,
        "BEP_side": BEP_side,
        "efficiency_drop_from_BEP": efficiency_drop_from_BEP,
        "efficiency_drop_pct": efficiency_drop_pct,
        "message": message,
    } 
def monitor_operating_points(results_df, bep_info):
    """Add status columns to results_df and print warnings for out-of-ROR windows."""
    statuses = [
        check_operating_status(q, eff, bep_info)
        for q, eff in zip(results_df["Q"], results_df["Efficiency"])
    ]

    out_df = results_df.copy()
    out_df["In_Safe_Envelope"] = [s["in_envelope"] for s in statuses]
    out_df["Distance_from_envelope_Q"] = [s["distance_Q"] for s in statuses]
    out_df["Distance_from_envelope_pct"] = [s["distance_pct"] for s in statuses]
    out_df["BEP_side"] = [s["BEP_side"] for s in statuses]
    out_df["Efficiency_drop_pct"] = [s["efficiency_drop_pct"] for s in statuses]
    out_df["Status_Message"] = [s["message"] for s in statuses]

    n_outside = (~out_df["In_Safe_Envelope"]).sum()
    n_nan = out_df["Q"].isna().sum()
    print(f"\nMonitoring: {n_outside} / {len(out_df)} windows are outside the safe envelope.")
    print(f"            {n_nan} windows had invalid Head (outside pump curve range).")
    for _, row in out_df[~out_df["In_Safe_Envelope"]].iterrows():
        print(f"  [t={row['Window_Start_s']:.0f}s] {row['Status_Message']}")

    return out_df
def plot_pump_curve_with_operating_points(Q, Head, Power, Efficiency, bep_info,
                                           results_df, window_number=None,
                                           time_interval=None, freq=None,
                                           save_path=None, show=True):
    """Plot pump curve with BEP, ROR and operating point(s).

    Units (display):
        - Q    : bbl/day
        - Head : psi
        - Eff  : %
        - Power: kW
    """
    sort_idx = np.argsort(Q)
    q_sorted = Q[sort_idx]
    head_sorted = Head[sort_idx]
    power_sorted = Power[sort_idx]
    eff_sorted = Efficiency[sort_idx]

    if window_number is not None:
        if window_number < 0 or window_number >= len(results_df):
            print(f"Warning: window_number {window_number} out of range. Using 0.")
            window_number = 0
        points = results_df.iloc[[window_number]]
    elif time_interval is not None:
        start_s, end_s = time_interval
        points = results_df[(results_df["Window_Start_s"] >= start_s) &
                             (results_df["Window_Start_s"] <= end_s)]
    else:
        points = results_df

    Q_low, Q_high, Q_BEP = bep_info["Q_low"], bep_info["Q_high"], bep_info["Q_BEP"]

    has_monitoring = ("In_Safe_Envelope" in points.columns
                      and "Status_Message" in points.columns)
    if has_monitoring:
        in_envelope = points["In_Safe_Envelope"]
    else:
        in_envelope = points["Q"].between(Q_low, Q_high)
    safe_pts = points[in_envelope]
    unsafe_pts = points[~in_envelope]

    fig = plt.figure(figsize=(15, 7))
    gs = fig.add_gridspec(1, 2, width_ratios=[3.0, 1.15], wspace=0.45,
                          left=0.12, right=0.97, top=0.90, bottom=0.15)
    ax_head = fig.add_subplot(gs[0, 0])
    ax_side = fig.add_subplot(gs[0, 1])

    ax_side.set_xticks([]); ax_side.set_yticks([])
    ax_side.set_xlim(0, 1); ax_side.set_ylim(0, 1)
    ax_side.set_frame_on(True)
    for spine in ax_side.spines.values():
        spine.set_edgecolor("#2C3E50")
        spine.set_linewidth(2.5)

    ax_head.axvspan(Q_low, Q_high, color="gold", alpha=0.25, zorder=0)
    ax_head.axvline(Q_BEP, color="crimson", linewidth=2, zorder=1)

    ax_head.plot(q_sorted, head_sorted, color="navy", linewidth=2)
    ax_head.set_xlabel("Capacity, Q (bbl/day)", fontsize=11, fontweight="bold")
    ax_head.set_xlim(0, 2500)
    ax_head.set_ylabel("Head, psi", color="navy", fontsize=11, fontweight="bold")
    ax_head.tick_params(axis="y", colors="navy", labelsize=9)
    ax_head.tick_params(axis="x", labelsize=9)

    ax_eff = ax_head.twinx()
    ax_eff.spines["right"].set_visible(False)
    ax_eff.spines["left"].set_position(("axes", -0.10))
    ax_eff.spines["left"].set_visible(True)
    ax_eff.spines["left"].set_color("turquoise")
    ax_eff.spines["left"].set_linewidth(1.5)
    ax_eff.yaxis.set_label_position("left")
    ax_eff.yaxis.set_ticks_position("left")
    ax_eff.plot(q_sorted, eff_sorted, color="turquoise", linewidth=2)
    ax_eff.set_ylabel("Eff, %", color="turquoise", fontsize=11, fontweight="bold")
    ax_eff.tick_params(axis="y", colors="turquoise", labelsize=9)
    ax_eff.set_ylim(0, 100)

    ax_power = ax_head.twinx()
    ax_power.spines["right"].set_position(("axes", 1.10))
    ax_power.plot(q_sorted, power_sorted, color="steelblue", linewidth=2)
    ax_power.set_ylabel("Power, kW", color="steelblue", fontsize=11, fontweight="bold")
    ax_power.tick_params(axis="y", colors="steelblue", labelsize=9)
    ax_power.set_ylim(0, 100)

    fig.canvas.draw()
    pos = ax_head.get_position()
    border = Rectangle(
        (pos.x0 - 0.100, pos.y0 - 0.075),
        pos.width + 0.215,
        pos.height + 0.135,
        transform=fig.transFigure,
        facecolor="none", edgecolor="black",
        linewidth=2.5, zorder=0, clip_on=False,
    )
    fig.add_artist(border)

    for pts, edge_color, label in [
        (safe_pts, "black", "Operating point (safe)"),
        (unsafe_pts, "darkred", "Operating point (outside safe range)"),
    ]:
        if len(pts) == 0:
            continue
        ax_eff.scatter(pts["Q"], pts["Efficiency"], s=80, facecolor="yellow",
                        edgecolor=edge_color, linewidth=1.8, zorder=5, label=label)

    if len(points) == 1:
        row = results_df.iloc[window_number] if window_number is not None else points.iloc[0]
        info_text = (
            f"Q = {row['Q']:.1f} bbl/day\n"
            f"Head = {row['Head']:.1f} psi\n"
            f"Eff = {row['Efficiency']:.1f} %\n"
            f"Power = {row['Power']:.1f} kW"
        )
        if "In_Safe_Envelope" in row.index:
            box_edge = "green" if row["In_Safe_Envelope"] else "darkred"
        else:
            box_edge = "red"
        ax_eff.annotate(
            info_text,
            xy=(row["Q"], row["Efficiency"]),
            xytext=(15, 15), textcoords="offset points",
            fontsize=9, fontweight="bold", color="black",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                      edgecolor=box_edge, linewidth=1.5, alpha=0.95),
            arrowprops=dict(arrowstyle="->", color=box_edge, linewidth=1.2),
            zorder=6,
        )

    title = "Electrical Submersible Pump Performance Curve Monitoring"
    if window_number is not None:
        w_start = results_df.iloc[window_number]["Window_Start_s"]
        title += f"\n(window #{window_number}, t = {w_start:.0f}s)"
    elif time_interval is not None:
        title += f"\n(t = {time_interval[0]:.0f}s to {time_interval[1]:.0f}s)"
    if freq is not None:
        title += f"  |  Freq = {freq:.1f} Hz"
    ax_head.set_title(title, fontsize=11, fontweight="bold")
    ax_head.grid(True, linestyle=":", alpha=0.5)

    handles, labels = ax_eff.get_legend_handles_labels()
    if handles:
        ax_eff.legend(handles, labels, loc="lower right", fontsize=8)

    # ---------- SIDEBAR KPI ----------
    def draw_kpi_card(ax, x, y, w, h, label, value, unit="",
                      value_color="#2C3E50", bg_color="#ECF0F1",
                      border_color="#BDC3C7"):
        card = FancyBboxPatch(
            (x + 0.005, y + 0.005), w - 0.01, h - 0.01,
            boxstyle="round,pad=0.01,rounding_size=0.02",
            transform=ax.transAxes,
            facecolor=bg_color, edgecolor=border_color,
            linewidth=1.5, zorder=2, clip_on=False,
        )
        ax.add_patch(card)
        ax.text(x + w/2, y + h*0.72, label, transform=ax.transAxes,
                ha="center", va="center", fontsize=8, fontweight="bold",
                color="#7F8C8D", zorder=3)
        value_str = f"{value}" + (f" {unit}" if unit else "")
        ax.text(x + w/2, y + h*0.30, value_str, transform=ax.transAxes,
                ha="center", va="center", fontsize=11, fontweight="bold",
                color=value_color, zorder=3)

    ax_side.text(0.5, 0.97, "ESP Performance Monitoring",
                 transform=ax_side.transAxes, ha="center", va="top",
                 fontsize=12, fontweight="bold", color="#2C3E50")
    ax_side.plot([0.06, 0.94], [0.915, 0.915], transform=ax_side.transAxes,
                 color="#2C3E50", linewidth=2.0, clip_on=False)

    if len(points) == 1 and has_monitoring:
        srow = results_df.iloc[window_number] if window_number is not None else points.iloc[0]
        is_safe = bool(srow["In_Safe_Envelope"])
        status_word = "SAFE" if is_safe else "WARNING"
        status_bg = "#D5F5E3" if is_safe else "#FADBD8"
        status_border = "#27AE60" if is_safe else "#C0392B"
        status_text_color = "#1E8449" if is_safe else "#922B21"

        draw_kpi_card(ax_side, 0.06, 0.78, 0.88, 0.11,
                      label="STATUS", value=status_word,
                      value_color=status_text_color,
                      bg_color=status_bg, border_color=status_border)

        wrapped_msg = "\n".join(textwrap.wrap(str(srow["Status_Message"]), width=38))
        ax_side.text(0.5, 0.745, wrapped_msg, transform=ax_side.transAxes,
                     ha="center", va="top", fontsize=7.5,
                     color="#2C3E50", linespacing=1.4)
        kpi_top = 0.62
    else:
        kpi_top = 0.88

    card_w, card_h, gap_y = 0.42, 0.085, 0.025
    col_x = [0.05, 0.53]

    def row_y(i, top):
        return top - (i + 1) * card_h - i * gap_y

    y0 = row_y(0, kpi_top)
    draw_kpi_card(ax_side, col_x[0], y0, card_w, card_h,
                  label="Q_BEP", value=f"{bep_info['Q_BEP']:.0f}", unit="bbl/day",
                  value_color="#1F618D", bg_color="#EAF2F8", border_color="#5DADE2")
    draw_kpi_card(ax_side, col_x[1], y0, card_w, card_h,
                  label="BEP Efficiency", value=f"{bep_info['Efficiency_BEP']:.1f}", unit="%",
                  value_color="#117A65", bg_color="#E8F8F5", border_color="#48C9B0")

    y1 = row_y(1, kpi_top)
    draw_kpi_card(ax_side, col_x[0], y1, card_w, card_h,
                  label="ROR Lower", value=f"{bep_info['Q_low']:.0f}", unit="bbl/day",
                  value_color="#7D6608", bg_color="#FCF3CF", border_color="#F1C40F")
    draw_kpi_card(ax_side, col_x[1], y1, card_w, card_h,
                  label="ROR Upper", value=f"{bep_info['Q_high']:.0f}", unit="bbl/day",
                  value_color="#7D6608", bg_color="#FCF3CF", border_color="#F1C40F")

    if len(points) == 1:
        prow = results_df.iloc[window_number] if window_number is not None else points.iloc[0]
        freq_str = f"{freq:.1f}" if freq is not None else "N/A"
        window_str = f"#{window_number}" if window_number is not None else "N/A"

        y2 = row_y(2, kpi_top)
        draw_kpi_card(ax_side, col_x[0], y2, card_w, card_h,
                      label="Q Operating", value=f"{prow['Q']:.0f}", unit="bbl/day",
                      value_color="#6C3483", bg_color="#F4ECF7", border_color="#A569BD")
        draw_kpi_card(ax_side, col_x[1], y2, card_w, card_h,
                      label="Eff Operating", value=f"{prow['Efficiency']:.1f}", unit="%",
                      value_color="#6C3483", bg_color="#F4ECF7", border_color="#A569BD")

        y3 = row_y(3, kpi_top)
        draw_kpi_card(ax_side, col_x[0], y3, card_w, card_h,
                      label="Head Operating", value=f"{prow['Head']:.1f}", unit="psi",
                      value_color="#154360", bg_color="#D6EAF8", border_color="#2E86C1")
        draw_kpi_card(ax_side, col_x[1], y3, card_w, card_h,
                      label="Power Operating", value=f"{prow['Power']:.1f}", unit="kW",
                      value_color="#154360", bg_color="#D6EAF8", border_color="#2E86C1")

        y4 = row_y(4, kpi_top)
        draw_kpi_card(ax_side, col_x[0], y4, card_w, card_h,
                      label="Frequency", value=freq_str, unit="Hz",
                      value_color="#7B241C", bg_color="#FADBD8", border_color="#E74C3C")
        draw_kpi_card(ax_side, col_x[1], y4, card_w, card_h,
                      label="Window", value=window_str, unit="",
                      value_color="#7B241C", bg_color="#FADBD8", border_color="#E74C3C")
    else:
        y2 = row_y(2, kpi_top)
        freq_str = f"{freq:.1f}" if freq is not None else "N/A"
        draw_kpi_card(ax_side, col_x[0], y2, card_w, card_h,
                      label="Frequency", value=freq_str, unit="Hz",
                      value_color="#7B241C", bg_color="#FADBD8", border_color="#E74C3C")
        draw_kpi_card(ax_side, col_x[1], y2, card_w, card_h,
                      label="Points Shown", value=f"{len(points)}", unit="",
                      value_color="#7B241C", bg_color="#FADBD8", border_color="#E74C3C")

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved chart to {save_path}")

    if show:
        plt.show()

    return fig
def save_results(results_df, out_path):
    """Save results DataFrame to CSV or Excel depending on extension."""
    ext = os.path.splitext(out_path)[1].lower()
    if ext == ".csv":
        results_df.to_csv(out_path, index=False)
    elif ext in (".xlsx", ".xls"):
        results_df.to_excel(out_path, index=False)
    else:
        results_df.to_csv(out_path, index=False)
    print(f"\nSaved {len(results_df)} rows to {out_path}")
def plot_tornado_family(Q_base, Head_base, Power_base=None,
                         base_freq=57.3,
                         freqs=(40, 45, 50, 55, 60, 65, 70, 75, 80),
                         bep_info=None,
                         operating_point=None,
                         freq=None,
                         xlim=None,
                         ylim=None,
                         title_extra="",
                         save_path=None,
                         show=True):
    """Plot tornado family with frequency + power label at start of each curve.

    Units (display):
        - Q    : bbl/day
        - Head : psi
        - Power: kW

    Affinity laws:
        Q_f = Q_base * (f / f_base)
        H_f = H_base * (f / f_base)^2
        P_f = P_base * (f / f_base)^3

    ROR (Recommended Operating Range):
        For each frequency f, Q_low and Q_high are scaled by affinity
        (Q_low_f = Q_low_base * ratio, Q_high_f = Q_high_base * ratio).
        The corresponding Head on each frequency curve is interpolated
        and marked with an orange marker. The locus of these markers
        across frequencies is drawn as two dotted orange lines.
    """
    fig, ax = plt.subplots(figsize=(11, 7.5))

    # ---- Sort base curve by Q ----
    sort_idx = np.argsort(Q_base)
    Q_base_sorted = np.asarray(Q_base)[sort_idx]
    Head_base_sorted = np.asarray(Head_base)[sort_idx]

    if Power_base is not None:
        Power_base_sorted = np.asarray(Power_base)[sort_idx]
    else:
        Power_base_sorted = None

    # ---- Prepare BEP base + ROR base ----
    if bep_info is not None:
        Q_BEP_base = float(bep_info["Q_BEP"])
        Head_BEP_base = float(np.interp(Q_BEP_base,
                                        Q_base_sorted, Head_base_sorted))
        Q_low_base  = float(bep_info["Q_low"])
        Q_high_base = float(bep_info["Q_high"])
    else:
        Q_BEP_base = None
        Head_BEP_base = None
        Q_low_base = None
        Q_high_base = None

    # =========================================================
    # 1) Frequency curves + labels + BEP + ROR markers
    # =========================================================
    for f in freqs:
        ratio = f / base_freq
        Q_f = Q_base_sorted * ratio
        H_f = Head_base_sorted * (ratio ** 2)

        # Head-Q curve
        ax.plot(Q_f, H_f, color="crimson", linewidth=1.2, alpha=0.85)

        # ---- Combined label: frequency + power at start of curve ----
        if Power_base_sorted is not None:
            P_start_base = Power_base_sorted[0]
            P_start_f = P_start_base * (ratio ** 3)
            label_txt = f"{f:.0f} Hz  |  P = {P_start_f:.0f} kW"
        else:
            label_txt = f"{f:.0f} Hz"

        ax.annotate(
            label_txt,
            xy=(Q_f[0], H_f[0]),
            xytext=(4, 4), textcoords="offset points",
            fontsize=8.5, color="black", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.25",
                      fc="white", ec="lightgray",
                      linewidth=0.6, alpha=0.85),
            zorder=5,
        )

        # ---- BEP marker ----
        if Q_BEP_base is not None:
            Q_BEP_f = Q_BEP_base * ratio
            H_BEP_f = float(np.interp(Q_BEP_f, Q_f, H_f))
            ax.scatter([Q_BEP_f], [H_BEP_f],
                       marker="o", s=18, facecolor="darkgreen",
                       edgecolor="black", linewidth=0.8, zorder=4)

        # ---- ROR markers (Q_low and Q_high scaled by affinity) ----
        if Q_low_base is not None:
            Q_low_f  = Q_low_base  * ratio
            Q_high_f = Q_high_base * ratio

            # Only mark if inside this frequency curve's Q range
            if Q_f[0] <= Q_low_f <= Q_f[-1]:
                H_low_f = float(np.interp(Q_low_f, Q_f, H_f))
                ax.scatter([Q_low_f], [H_low_f],
                           marker="|", s=120, color="orange",
                           linewidth=2.2, zorder=4)

            if Q_f[0] <= Q_high_f <= Q_f[-1]:
                H_high_f = float(np.interp(Q_high_f, Q_f, H_f))
                ax.scatter([Q_high_f], [H_high_f],
                           marker="|", s=120, color="orange",
                           linewidth=2.2, zorder=4)

    # =========================================================
    # 2) BEP locus across frequencies
    # =========================================================
    if Q_BEP_base is not None:
        bep_Qs, bep_Hs = [], []
        for f in freqs:
            ratio = f / base_freq
            bep_Qs.append(Q_BEP_base * ratio)
            bep_Hs.append(Head_BEP_base * (ratio ** 2))

        ax.plot(bep_Qs, bep_Hs, linestyle="--", color="black",
                linewidth=1.8, label="BEP locus", zorder=6)

    # =========================================================
    # 3) ROR locus across frequencies (line connecting markers)
    # =========================================================
    if Q_low_base is not None:
        ror_low_Qs,  ror_low_Hs  = [], []
        ror_high_Qs, ror_high_Hs = [], []

        for f in freqs:
            ratio = f / base_freq
            Q_f = Q_base_sorted * ratio
            H_f = Head_base_sorted * (ratio ** 2)

            Q_low_f  = Q_low_base  * ratio
            Q_high_f = Q_high_base * ratio

            if Q_f[0] <= Q_low_f <= Q_f[-1]:
                ror_low_Qs.append(Q_low_f)
                ror_low_Hs.append(float(np.interp(Q_low_f, Q_f, H_f)))

            if Q_f[0] <= Q_high_f <= Q_f[-1]:
                ror_high_Qs.append(Q_high_f)
                ror_high_Hs.append(float(np.interp(Q_high_f, Q_f, H_f)))

        if len(ror_low_Qs) >= 2:
            ax.plot(ror_low_Qs, ror_low_Hs,
                    linestyle=":", color="orange", linewidth=1.8,
                    label="ROR lower", zorder=6)

        if len(ror_high_Qs) >= 2:
            ax.plot(ror_high_Qs, ror_high_Hs,
                    linestyle=":", color="orange", linewidth=1.8,
                    label="ROR upper", zorder=6)

    # =========================================================
    # 4) Operating point
    # =========================================================
    if operating_point is not None:
        q_op = float(operating_point["Q"])
        h_op = float(operating_point["Head"])
        if np.isfinite(q_op) and np.isfinite(h_op):
            ax.scatter([q_op], [h_op], s=160, facecolor="yellow",
                       edgecolor="black", linewidth=2, zorder=7,
                       label="Operating point")

            p_op = operating_point.get("Power", None)
            if p_op is not None and np.isfinite(p_op):
                label_txt = f"Q={q_op:.0f}, H={h_op:.0f}, P={float(p_op):.0f} kW"
            else:
                label_txt = f"Q={q_op:.0f}, H={h_op:.0f}"

            ax.annotate(
                label_txt,
                xy=(q_op, h_op),
                xytext=(12, 12), textcoords="offset points",
                fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.4", fc="white",
                          ec="black", linewidth=1),
                arrowprops=dict(arrowstyle="->", color="black"),
            )

    # =========================================================
    # 5) Labels, limits, title, legend
    # =========================================================
    ax.set_xlabel("Capacity, Q (bbl/day)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Head, H (psi)", fontsize=11, fontweight="bold")

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    title = "Multi-Frequency Pump Curve (Tornado Plot)"
    if freq is not None:
        title += f"  |  Operating freq = {freq:.1f} Hz"
    if bep_info is not None:
        title += (f"  |  Q_BEP = {bep_info['Q_BEP']:.0f} bbl/day"
                  f"  |  ROR = [{bep_info['Q_low']:.0f}, "
                  f"{bep_info['Q_high']:.0f}]")
    if title_extra:
        title += f"\n{title_extra}"
    ax.set_title(title, fontsize=12, fontweight="bold")

    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(loc="best", fontsize=9)

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved tornado chart to {save_path}")

    if show:
        plt.show()

    return fig
def main():
    # ---------------- Paths ----------------
    sensor_data_path = r"D:\ESP Project\JR-09 data\Data\2026.09.14  DOWNLOAD VSD DATA.xlsx"
    pump_curve_path  = r"D:\ESP Project\JR-09 data\TDH.xlsx"

    out_path     = r"D:\ESP Project\Python codes\ESP_rate_calc\results\operating_point_report.xlsx"
    plot_path    = r"D:\ESP Project\Python codes\ESP_rate_calc\results\operating_plot.png"
    tornado_path = r"D:\ESP Project\Python codes\ESP_rate_calc\results\tornado_plot.png"

    # ---------------- Settings ----------------
    pump_freq        = 57.3      # reference frequency of digitized curve (Hz)
    observe_window   = 1.0       # seconds
    head_match_tol   = 1e-6
    N_stages         = 1         # curve already represents total pump head
    curve_is_in_feet = True      # set True if Excel Head is in feet
    SG_default       = 0.9       # reference SG of the digitized curve

    # ---------------- Load sensor + curve data ----------------
    # NOTE: 'Rate' column in the Excel is ALREADY in bbl/day.
    (Time, Elapsed_time, PIP_signal, PDP_signal,
     pump_Q, pump_Head, pump_Power, pump_Efficiency) = load_sensor_and_curve_data(
        sensor_data_path, pump_curve_path
    )

    print("=" * 60)
    print("RAW digitized curve (before any scaling):")
    print(f"  Q          range: {np.nanmin(pump_Q):.2f} to {np.nanmax(pump_Q):.2f} bbl/day")
    print(f"  Head       range: {np.nanmin(pump_Head):.2f} to {np.nanmax(pump_Head):.2f} ft")
    print(f"  Power      range: {np.nanmin(pump_Power):.2f} to {np.nanmax(pump_Power):.2f} kW")
    print(f"  Efficiency range: {np.nanmin(pump_Efficiency):.2f} to {np.nanmax(pump_Efficiency):.2f} %")
    print("=" * 60)

    # ---------------- Unit conversion for Head (ft -> m) ----------------
    pump_Head = pump_Head * N_stages
    if curve_is_in_feet:
        pump_Head = pump_Head * 0.3048

    print("Curve after N_stages + unit conversion:")
    print(f"  Head range (m): {np.nanmin(pump_Head):.2f} to {np.nanmax(pump_Head):.2f}")

    # ---------------- Fluid SG (user input with default) ----------------
    SG_input = input(f"Please enter fluid specific gravity SG "
                     f"(default = {SG_default}): ").strip()
    fluid_SG = float(SG_input) if SG_input else SG_default
    print(f"Using fluid SG = {fluid_SG} "
          f"({'default' if not SG_input else 'user-provided'})")

    # ---------------- Frequency & affinity ----------------
    Freq = float(input("Please enter pump operating frequency (Hz): "))
    f_ratio = Freq / pump_freq
    print(f"Frequency ratio: {pump_freq} -> {Freq} Hz  (ratio = {f_ratio:.3f})")

    # Affinity scaling on Q (bbl/day), Head (m), Power (kW)
    Q_af, Head_af, Power_af = apply_affinity_scaling(
        pump_Q, pump_Head, pump_Power, f_ratio
    )

    # ---------------- SG correction on Power ----------------
    Power_af = apply_SG_correction_to_power(Power_af, fluid_SG,
                                            SG_reference=SG_default)
    print(f"Power corrected for SG={fluid_SG} (reference SG={SG_default}).")
    print(f"  Power range after affinity + SG: "
          f"{np.nanmin(Power_af):.2f} to {np.nanmax(Power_af):.2f} kW")

    # Efficiency is unchanged by affinity laws
    Efficiency_curve = pump_Efficiency.copy()

    # ---------------- Build interpolation arrays ----------------
    head_arr_for_q, q_arr_for_q, q_arr, eff_arr, power_arr = build_interp_arrays(
        Q_af, Head_af, Power_af, Efficiency_curve
    )

    # ---------------- Compute operating points ----------------
    results_df = compute_operating_points(
        Elapsed_time, PIP_signal, PDP_signal,
        head_arr_for_q, q_arr_for_q, q_arr, eff_arr, power_arr,
        observe_window=observe_window,
        SG=fluid_SG,
        head_match_tol=head_match_tol,
        pressure_already_in_meters=False,
    )

    print("\nResults (first 10 rows) — internal units:")
    print(results_df[["Window_Start_s", "PIP", "PDP", "Head",
                       "Q", "Efficiency", "Power", "Status"]].head(10))

    # ---------------- BEP & ROR (Q in bbl/day) ----------------
    bep_info = find_bep_and_safe_envelope(Q_af, Efficiency_curve)

    print("\nBEP Information (at user frequency):")
    print(f"  Q_BEP          = {bep_info['Q_BEP']:.2f} bbl/day")
    print(f"  Efficiency_BEP = {bep_info['Efficiency_BEP']:.2f} %")
    print(f"  ROR            = [{bep_info['Q_low']:.2f}, {bep_info['Q_high']:.2f}] bbl/day")

    # BEP at base frequency (for tornado)
    bep_info_base = find_bep_and_safe_envelope(pump_Q, pump_Efficiency)

    # ---------------- Monitoring ----------------
    results_df = monitor_operating_points(results_df, bep_info)

    # =================================================================
    # Convert ONLY HEAD to psi for plotting.
    # Q is already in bbl/day — no conversion needed.
    # =================================================================

    Q_af_bbl     = Q_af                              # already bbl/day
    Head_af_psi  = head_meters_to_psi(Head_af, fluid_SG)

    pump_Q_bbl     = pump_Q                          # already bbl/day
    pump_Head_psi  = head_meters_to_psi(pump_Head, fluid_SG)

    bep_info_disp      = dict(bep_info)
    bep_info_base_disp = dict(bep_info_base)

    results_disp = results_df.copy()
    results_disp["Head"] = head_meters_to_psi(results_df["Head"], fluid_SG)

    # Power base for tornado (SG-corrected, no affinity)
    Power_base_for_tornado = apply_SG_correction_to_power(
        pump_Power, fluid_SG, SG_reference=SG_default
    )

    # ---------------- Save results ----------------
    save_results(results_disp, out_path)

    # ---------------- Main monitoring plot ----------------
    window_idx = int(input(
        f"\nEnter the observation window number to plot "
        f"(0 to {len(results_disp) - 1}, each window = {observe_window}s): "
    ))

    plot_pump_curve_with_operating_points(
        Q_af_bbl, Head_af_psi, Power_af, Efficiency_curve,
        bep_info_disp, results_disp,
        window_number=window_idx,
        freq=Freq,
        save_path=plot_path,
    )

    # ---------------- Tornado plot ----------------
    op_point = results_disp.iloc[window_idx]

    plot_tornado_family(
        pump_Q_bbl, pump_Head_psi, Power_base_for_tornado,
        base_freq=pump_freq,
        freqs=(40, 45, 50, 55, 60, 65, 70, 75, 80),
        bep_info=bep_info_base_disp,
        operating_point=op_point,
        freq=Freq,
        xlim=(0, 2500),
        ylim=(0, 2500),
        title_extra=f"window #{window_idx}, t = {op_point['Window_Start_s']:.0f}s",
        save_path=tornado_path,
    )


if __name__ == "__main__":
    main()

RAW digitized curve (before any scaling):
  Q          range: 0.00 to 2400.00 bbl/day
  Head       range: 0.00 to 3450.40 ft
  Power      range: 20.60 to 38.00 kW
  Efficiency range: 0.00 to 65.50 %
Curve after N_stages + unit conversion:
  Head range (m): 0.00 to 1051.68
Using fluid SG = 0.9 (user-provided)
Frequency ratio: 57.3 -> 45.0 Hz  (ratio = 0.785)
Power corrected for SG=0.9 (reference SG=0.9).
  Power range after affinity + SG: 9.98 to 18.41 kW

Results (first 10 rows) — internal units:
   Window_Start_s          PIP          PDP        Head            Q  \
0             0.0  1787.204778  2081.609778  294.405000  1445.487503   
1            60.0  1787.204778  2081.495000  294.290222  1445.707859   
2           120.0  1794.321000  2081.265444  286.944444  1459.488729   
3           180.0  1792.943667  2081.495000  288.551333  1456.547036   
4           240.0  1791.451556  2081.609778  290.158222  1453.605343   
5           300.0  1787.204778  2081.839333  294.634556  1445.0467